In [1]:
from pinecone import Pinecone
import pickle
import tqdm
from tqdm import tqdm
from embed_model import embed_text
import pandas as pd

pc = Pinecone(api_key="pcsk_2xwBFb_CfbzWGVP9bZ4v1HNSY2y8oSRwrunFBF63ejGfoKxZosgfzK7YvQJdYuoniewFzJ")

c:\Users\PC\anaconda3\envs\myenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("foody_combined_data_final.csv")

In [3]:
df = df.fillna("")

for col in df.columns:
    df[col] = df[col].astype(str)

df["RestaurantID"] = df["RestaurantID"].apply(lambda x: str(int(float(x))))

In [4]:
def row_to_text(row):
    restaurant_ID = row["RestaurantID"]
    name = row["Name"]
    address = row["Address"]
    district = row["District"]
    area = row["Area"]

    price_min = row["PriceMin"]
    price_max = row["PriceMax"]

    meta_keywords = row["MetaKeywords"]
    cuisines = row["Cuisines"]
    target_audience = row["LstTargetAudience"]
    category = row["LstCategory"]
    guild = row["AccessGuide"] if row["AccessGuide"] != "" else "Không xác định"
    # restaurant_url = row["RestaurantUrl"]

    location_rate = row["Vị trí"]
    price_rate = row["Giá cả"]
    quality_rate = row["Chất lượng"]
    service_rate = row["Phục vụ"]

    space_rate = row["Không gian"]
    excellent = row["Excellent"]
    good = row["Good"]
    average = row["Average"]
    bad = row["Bad"]

    has_booking = row["HasBooking"] if row["HasBooking"] != "" else "Không xác định"
    has_delivery = row["HasDelivery"] if row["HasDelivery"] != "" else "Không xác định"
    has_promotion = row["HasPromotion"] if row["HasPromotion"] != "" else "Không xác định"
    total_view = row["TotalView"]
    total_favorite = row["TotalFavourite"]
    total_checkin = row["TotalCheckins"]

    home_delivery = "Có giao hàng tận nơi" if row["Giao tận nơi"] == "True" else "Không có giao hàng tận nơi"
    oder_table = "Có đặt bàn" if row["Đặt bàn"] == "True" else "Không có đặt bàn"
    return f"""
    Nhà hàng: {name}
    Địa chỉ: {address}, {district}, {area}
    ID nhà hàng: {restaurant_ID}

    Giá: từ {price_min} đến {price_max} VND

    Mô tả: {meta_keywords}
    Ẩm thực: {cuisines}
    Đối tượng mục tiêu: {target_audience}
    Danh mục: {category}
    Hướng dẫn đi lại: {guild}


    Đánh giá:
    - Vị trí: {location_rate}
    - Giá cả: {price_rate}
    - Chất lượng: {quality_rate}
    - Phục vụ: {service_rate}
    - Không gian: {space_rate}

    Số lượng đánh giá:
    - Excellent: {excellent}
    - Good: {good}
    - Average: {average}
    - Bad: {bad}

    Dịch vụ:
    - Giao tận nơi: {home_delivery}
    - Đặt bàn: {oder_table}

    Tổng lượt xem: {total_view}
    Tổng lượt yêu thích: {total_favorite}
    Tổng lượt check-in: {total_checkin}

    Có đặt bàn: {has_booking}
    Có giao hàng: {has_delivery}
    Có khuyến mãi: {has_promotion}

    """

In [5]:
# Nhân
# start = 0
# end = 30000
# texts = [row_to_text(row) for _, row in df.iloc[start:end].iterrows()]

start = 6720
end = 9600
texts = [row_to_text(row) for _, row in df.iloc[start:end].iterrows()]

# Đạt
# start = 30000
# end = 65008
# texts = [row_to_text(row) for _, row in df.iloc[start:end].iterrows()]

# Huy
# start = 65008
# end = len(df)
# texts = [row_to_text(row) for _, row in df.iloc[start:end].iterrows()]

In [6]:
index = pc.Index("restaurant-recommendation")

In [7]:
def clean_text(text):
    return text.encode("utf-8", "ignore").decode("utf-8")

In [8]:
batch_size = 16

for i in tqdm(range(0, len(texts), batch_size)):
    batch_names = []
    batch_ids = []
    batch_texts = []

    for j in range(i, min(i + batch_size, len(texts))):
        row = df.iloc[j + start]

        text = row_to_text(row)
        batch_texts.append(text)

        batch_names.append(row["Name"])
        batch_ids.append(row["RestaurantID"])
    
    if len(batch_names) == 0:
        continue 

    batch_emb = embed_text(batch_texts).cpu().numpy()

    vectors = [
        {
            "id": batch_ids[k],
            "values": batch_emb[k].tolist(),
            "metadata": {
                "text": clean_text("Nhà hàng: " + batch_names[k]),
            }
        }
        for k in range(len(batch_ids))
    ]

    index.upsert(vectors=vectors)

    del batch_emb, vectors

100%|██████████| 180/180 [2:23:14<00:00, 47.75s/it]  
